# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = **one content item for one client** (`content_id` + `client_id`), aggregated over a **fixed trailing 90-day window as of the starter export date** — same grain as w02, no daily time-series here. `impressions_last_30d` / `impressions_prev_30d` give me two non-overlapping 30-day sub-windows inside that 90-day span, which I use below to sanity-check the window math rather than treat as independent.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].copy()

print("grain check - one row per content_id:", qualified.content_id.is_unique)
print("rows in this window:", len(qualified))


grain check - one row per content_id: True
rows in this window: 30000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** `avg_position`, `ctr`, `impressions_90d`, `clicks_90d`, `word_count`, `content_age_days`, `days_since_last_update`, `scroll_rate`, `engagement_rate`, `content_type`, `main_intent`, `competition_level`, `freshness_tier` — all observed signals or transparent derived buckets, known before the review decision.

**Label:** `ctr_gap` (from w02) — `expected_ctr(avg_position) - ctr`, a defined-rule proxy built only from `avg_position` and `ctr`.

**Context (not features):** `content_id`, `client_id` (join/grouping keys only — mean nothing as values), `trend_direction` (a precalculated tier that overlaps conceptually with my CTR-based label, so I'm keeping it as a comparison point, not feeding it in and risking a circular result).

**Excluded, with why:**
- `char_count` — 0.94 correlation with `word_count`; redundant, dropping the noisier one.
- `provider_used`, `model_used` — describe how content was *produced*, not a search/engagement signal, and mostly missing anyway (71.5% / 19.1% null).
- `ai_sessions_90d` — only 6.4% of rows have any AI session at all; too sparse to be a real feature for this lane, matches the guide's own warning about this field.

In [2]:
feat_cols = ["avg_position", "ctr", "impressions_90d", "clicks_90d", "word_count", "char_count",
             "content_age_days", "days_since_last_update", "scroll_rate", "engagement_rate",
             "content_type", "main_intent", "competition_level", "freshness_tier",
             "ai_sessions_90d", "provider_used", "model_used", "trend_direction"]

print("missing rate per candidate field:")
print(qualified[feat_cols].isna().mean().round(3).sort_values(ascending=False))
print()
print("word_count vs char_count correlation:", round(qualified[["word_count", "char_count"]].corr().iloc[0, 1], 3))
print("pct rows with ai_sessions_90d > 0:", round((qualified.ai_sessions_90d > 0).mean() * 100, 2))


missing rate per candidate field:
provider_used             0.715
char_count                0.257
word_count                0.257
model_used                0.191
competition_level         0.087
main_intent               0.079
scroll_rate               0.004
avg_position              0.000
ctr                       0.000
clicks_90d                0.000
engagement_rate           0.000
days_since_last_update    0.000
content_age_days          0.000
impressions_90d           0.000
freshness_tier            0.000
content_type              0.000
ai_sessions_90d           0.000
trend_direction           0.000
dtype: float64

word_count vs char_count correlation: 0.939
pct rows with ai_sessions_90d > 0: 6.43


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Two checks not already covered in sections 1-2: the 30-day sub-window math actually adds up inside the 90-day total (no double-counting or inconsistency), and client representation isn't wildly skewed in a way that would let one client dominate the score.

In [3]:
# window consistency: last_30d + prev_30d should never exceed the 90d total
qualified["sum_30d_windows"] = qualified.impressions_last_30d.fillna(0) + qualified.impressions_prev_30d.fillna(0)
bad_windows = (qualified["sum_30d_windows"] > qualified.impressions_90d).mean()
print("pct rows where last30d+prev30d > 90d total (should be 0):", round(bad_windows * 100, 2))

# client concentration
vc = qualified.client_id.value_counts()
print("\nunique clients:", qualified.client_id.nunique())
print("top client's share of rows:", round(vc.iloc[0] / len(qualified) * 100, 1), "%")
print("top 5 clients' share of rows:", round(vc.iloc[:5].sum() / len(qualified) * 100, 1), "%")


pct rows where last30d+prev30d > 90d total (should be 0): 0.0

unique clients: 32
top client's share of rows: 23.4 %
top 5 clients' share of rows: 56.8 %


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Single snapshot, no time-series.** The starter CSV is one fixed 90-day window — it can't tell me whether a page's CTR improved or declined *after* a review, so it can't support a future-outcome label. That's a warehouse-only (`fact_content_daily_performance`) capability I don't have here; upgrading `ctr_gap` from a rule-based proxy to an observed-outcome label needs that table.
- **Client concentration is real, not edge case.** Just 32 clients back this 30,000-row set, and the top 5 alone account for 56.8% of rows. Any model trained on this needs a client-holdout split, not a plain random split — otherwise it's mostly memorizing a handful of clients' patterns.
- **`content_type`/`main_intent`/`competition_level` are unevenly missing** (up to ~26% for word/char count) — any feature built on them needs a documented "unknown" handling, not silent drop, or I'll quietly bias the sample toward whichever content type happens to report those fields.
- **GSC-only early history and `gsc_data_start`/`ga4_data_start` tracking gaps are documented in the guide's warehouse manifest, not something I've verified myself** — I don't have gated access to `dim_clients` in this environment, so that limit is cited from the guide, not confirmed by a query here. Flagging the difference honestly rather than presenting it as something I checked.
- **`expected_ctr` is still my own benchmark curve** (from w02), not derived from this data — so `ctr_gap` inherits that assumption and can't yet be defended as "the model discovered the right expected CTR," only "the model ranks by gap against my stated benchmark."

In [4]:
# concrete evidence for the missing-value and concentration limits above
print("missing pct - content_type/main_intent/competition_level/word_count:")
print(qualified[["content_type", "main_intent", "competition_level", "word_count"]].isna().mean().round(3))

print("\nclient row-count spread (min/median/max):",
      vc.min(), "/", int(vc.median()), "/", vc.max())


missing pct - content_type/main_intent/competition_level/word_count:
content_type         0.000
main_intent          0.079
competition_level    0.087
word_count           0.257
dtype: float64

client row-count spread (min/median/max): 3 / 567 / 7008


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.